# Data inventory

**Alwar groundwater study · 14-block domain (LGD 574–587)**

What is on Drive under `gw_framework/data`: every NetCDF, table, raster and shapefile, with metadata. Read-only, so it is safe to rerun at any time.

| | |
|---|---|
| **Reads** | `/content/drive/MyDrive/gw_framework/data/` |
| **Writes** | `gw_framework/data_inventory_report/` |
| **Cells** | 11 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. File inventory
- 2. Quick look at groundwater and CMIP6 tables

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["xarray", "netCDF4", "rasterio", "openpyxl", "geopandas", "pyarrow"], modflow=False)
del TOKEN, URL


## 1. File inventory

Run in order: cell 1 sets `ROOT` and `REPORT_DIR`, and the next cells use them.

**[orig #0]** Data inventory set-up: paths and report folder (`gw_framework/data_inventory_report`)  
<sub>✅ ran OK</sub>

In [ ]:
%pip -q install xarray netCDF4 rasterio openpyxl

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import re
import json
import warnings
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/gw_framework/data")
REPORT_DIR = ROOT.parent / "data_inventory_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not ROOT.exists():
    raise FileNotFoundError(
        f"Folder not found: {ROOT}\n"
        "Check the Google Drive folder name and path."
    )

print("Scanning root:", ROOT)
print("Report folder:", REPORT_DIR)

**[orig #65]** Inventory every supported file under `gw_framework/data`  
<sub>✅ ran OK</sub>

In [ ]:
SUPPORTED = {
    ".nc", ".nc4", ".cdf",       # NetCDF
    ".csv", ".tsv", ".txt",      # Tables
    ".tif", ".tiff", ".vrt",     # Rasters
    ".shp", ".gpkg", ".geojson", # GIS vectors
    ".json", ".parquet",
    ".xlsx", ".xls",
    ".zip"
}

files_found = []

for folder, dirs, filenames in os.walk(ROOT):
    # Avoid scanning generated reports
    dirs[:] = [
        d for d in dirs
        if Path(folder, d).resolve() != REPORT_DIR.resolve()
    ]

    for filename in filenames:
        p = Path(folder) / filename
        if p.name.startswith("."):
            continue

        try:
            size_bytes = p.stat().st_size
            files_found.append({
                "folder": str(p.parent.relative_to(ROOT)),
                "filename": p.name,
                "extension": p.suffix.lower(),
                "size_MB": round(size_bytes / (1024**2), 3),
                "size_bytes": size_bytes,
                "supported_for_inspection": p.suffix.lower() in SUPPORTED,
                "full_path": str(p)
            })
        except OSError as e:
            print("Could not access:", p, e)

inventory = pd.DataFrame(files_found)

if inventory.empty:
    print("No files found.")
else:
    display(inventory)

    inventory.to_csv(
        REPORT_DIR / "all_files_inventory.csv",
        index=False
    )

    print("\nTotal files:", len(inventory))
    print("\nFiles by folder:")
    display(
        inventory.groupby("folder")
        .agg(
            file_count=("filename", "count"),
            total_size_MB=("size_MB", "sum")
        )
        .sort_values("file_count", ascending=False)
        .reset_index()
    )

    print("\nFiles by extension:")
    display(
        inventory.groupby("extension")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

**[orig #66]** NetCDF metadata (variables, time span, bounds)  
<sub>✅ ran OK</sub>

In [ ]:
import xarray as xr

nc_records = []

nc_files = inventory[
    inventory["extension"].isin([".nc", ".nc4", ".cdf"])
]["full_path"].tolist()

for path in nc_files:
    try:
        with xr.open_dataset(path, decode_times=True) as ds:
            record = {
                "file": Path(path).name,
                "folder": str(Path(path).parent.relative_to(ROOT)),
                "variables": ", ".join(ds.data_vars),
                "dimensions": json.dumps(dict(ds.sizes)),
                "coordinates": ", ".join(ds.coords),
                "global_attributes": json.dumps(
                    {k: str(v)[:200] for k, v in ds.attrs.items()}
                )
            }

            # Extract variable metadata
            var_info = []
            for name, da in ds.data_vars.items():
                var_info.append({
                    "name": name,
                    "units": da.attrs.get("units", ""),
                    "long_name": da.attrs.get("long_name", ""),
                    "dimensions": str(da.dims),
                    "dtype": str(da.dtype)
                })

            record["variable_details"] = json.dumps(var_info)
            record["time_start"] = ""
            record["time_end"] = ""
            record["time_count"] = ""

            # Find time coordinate
            time_name = next(
                (n for n in ["time", "valid_time", "date"]
                 if n in ds.coords),
                None
            )

            if time_name:
                time_values = ds[time_name].values
                if len(time_values):
                    record["time_start"] = str(time_values[0])
                    record["time_end"] = str(time_values[-1])
                    record["time_count"] = len(time_values)

            # Coordinate bounds
            for coord in ["lat", "latitude", "lon", "longitude", "x", "y"]:
                if coord in ds.coords:
                    vals = ds[coord].values
                    if vals.size:
                        record[f"{coord}_min"] = float(np.nanmin(vals))
                        record[f"{coord}_max"] = float(np.nanmax(vals))

            nc_records.append(record)

            print("\nNETCDF:", path)
            print("Variables:", record["variables"])
            print("Dimensions:", record["dimensions"])
            print("Time:", record["time_start"], "to", record["time_end"])

    except Exception as e:
        nc_records.append({
            "file": Path(path).name,
            "folder": str(Path(path).parent.relative_to(ROOT)),
            "error": str(e)
        })
        print("NETCDF ERROR:", path, e)

nc_report = pd.DataFrame(nc_records)
display(nc_report)

nc_report.to_csv(
    REPORT_DIR / "netcdf_metadata.csv",
    index=False
)

**[orig #67]** Table files (CSV/Excel/parquet): columns and likely date fields  
<sub>✅ ran OK</sub>

In [ ]:
table_records = []

table_files = inventory[
    inventory["extension"].isin([".csv", ".tsv", ".txt", ".xlsx", ".xls"])
]["full_path"].tolist()

for path in table_files:
    p = Path(path)

    try:
        if p.suffix.lower() in [".xlsx", ".xls"]:
            book = pd.ExcelFile(path)
            for sheet in book.sheet_names:
                sample = pd.read_excel(path, sheet_name=sheet, nrows=5)
                table_records.append({
                    "file": p.name,
                    "folder": str(p.parent.relative_to(ROOT)),
                    "sheet": sheet,
                    "columns": ", ".join(map(str, sample.columns)),
                    "sample": sample.head(3).to_json(
                        orient="records", date_format="iso"
                    ),
                    "scenario_or_model_in_filename": p.stem
                })
                print(f"\nEXCEL: {p.name} | Sheet: {sheet}")
                display(sample.head(3))
        else:
            sep = "\t" if p.suffix.lower() == ".tsv" else None
            sample = pd.read_csv(
                path, sep=sep, engine="python",
                nrows=5, on_bad_lines="skip"
            )

            # Look for likely date columns
            date_candidates = [
                c for c in sample.columns
                if any(k in str(c).lower()
                       for k in ["date", "time", "year"])
            ]

            table_records.append({
                "file": p.name,
                "folder": str(p.parent.relative_to(ROOT)),
                "sheet": "",
                "columns": ", ".join(map(str, sample.columns)),
                "date_candidates": ", ".join(map(str, date_candidates)),
                "sample": sample.head(3).to_json(
                    orient="records", date_format="iso"
                ),
                "scenario_or_model_in_filename": p.stem
            })

            print("\nTABLE:", path)
            print("Columns:", list(sample.columns))
            display(sample.head(3))

    except Exception as e:
        table_records.append({
            "file": p.name,
            "folder": str(p.parent.relative_to(ROOT)),
            "error": str(e)
        })
        print("TABLE ERROR:", path, e)

table_report = pd.DataFrame(table_records)
table_report.to_csv(
    REPORT_DIR / "table_metadata.csv",
    index=False
)

**[orig #68]** Rasters (GeoTIFF) metadata  
<sub>✅ ran OK</sub>

In [ ]:
import rasterio

raster_records = []

raster_files = inventory[
    inventory["extension"].isin([".tif", ".tiff", ".vrt"])
]["full_path"].tolist()

for path in raster_files:
    try:
        with rasterio.open(path) as src:
            row = {
                "file": Path(path).name,
                "folder": str(Path(path).parent.relative_to(ROOT)),
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "bands": src.count,
                "pixel_size_x": src.transform.a,
                "pixel_size_y": src.transform.e,
                "xmin": src.bounds.left,
                "ymin": src.bounds.bottom,
                "xmax": src.bounds.right,
                "ymax": src.bounds.top,
                "nodata": src.nodata,
                "band_descriptions": str(src.descriptions)
            }
            raster_records.append(row)

            print("\nRASTER:", path)
            print("CRS:", src.crs, "| Bands:", src.count)
            print("Bounds:", src.bounds)

    except Exception as e:
        raster_records.append({
            "file": Path(path).name,
            "error": str(e)
        })
        print("RASTER ERROR:", path, e)

raster_report = pd.DataFrame(raster_records)
raster_report.to_csv(
    REPORT_DIR / "raster_metadata.csv",
    index=False
)

# Identify vector files and shapefile components
vector_files = inventory[
    inventory["extension"].isin([".shp", ".gpkg", ".geojson", ".json"])
][["folder", "filename", "full_path"]]

print("\nPotential study-area boundaries and vector datasets:")
display(vector_files)
vector_files.to_csv(
    REPORT_DIR / "vector_file_inventory.csv",
    index=False
)

**[orig #69]** Vector/shapefile + scenario-name scan; write text summary and zip the reports  
<sub>✅ ran OK</sub>

In [ ]:
import zipfile

scenario_pattern = re.compile(
    r"historical|ssp119|ssp126|ssp245|ssp370|ssp434|ssp460|ssp585",
    re.IGNORECASE
)

scenario_records = []

for _, row in inventory.iterrows():
    name = row["filename"]
    matches = scenario_pattern.findall(name)

    if matches or any(
        word in name.lower()
        for word in ["cmip6", "gwl", "groundwater", "precip", "tmax", "tmin"]
    ):
        scenario_records.append({
            "folder": row["folder"],
            "filename": name,
            "scenario_labels": ", ".join(sorted(set(
                x.lower() for x in matches
            ))),
            "size_MB": row["size_MB"]
        })

scenario_report = pd.DataFrame(scenario_records)
display(scenario_report)

scenario_report.to_csv(
    REPORT_DIR / "scenario_filename_scan.csv",
    index=False
)

# Create a text summary
summary_path = REPORT_DIR / "README_inventory_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("CMIP6 / GROUNDWATER DATA INVENTORY\n")
    f.write("=" * 45 + "\n")
    f.write(f"Root folder: {ROOT}\n")
    f.write(f"Total files: {len(inventory)}\n\n")

    f.write("FILES BY FOLDER\n")
    f.write("-" * 30 + "\n")
    if not inventory.empty:
        f.write(
            inventory.groupby("folder")
            .size()
            .to_string() + "\n\n"
        )

    f.write("FILES BY EXTENSION\n")
    f.write("-" * 30 + "\n")
    f.write(
        inventory.groupby("extension")
        .size()
        .to_string() + "\n\n"
    )

    f.write("NETCDF FILES\n")
    f.write("-" * 30 + "\n")
    if not nc_report.empty:
        f.write(nc_report.to_string(index=False) + "\n\n")

    f.write("TABLE FILES\n")
    f.write("-" * 30 + "\n")
    if not table_report.empty:
        f.write(table_report.to_string(index=False) + "\n\n")

# Bundle only the inspection reports, not the original data
report_zip = REPORT_DIR / "data_inventory_reports.zip"

with zipfile.ZipFile(report_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in REPORT_DIR.iterdir():
        if p.is_file() and p != report_zip:
            z.write(p, arcname=p.name)

print("\n" + "=" * 55)
print("SCAN COMPLETE")
print("Files scanned:", len(inventory))
print("NetCDF files:", len(nc_files))
print("Report directory:", REPORT_DIR)
print("ZIP report:", report_zip)
print("\nPlease download this ZIP and upload it into our chat:")
print(report_zip)

## 2. Quick look at groundwater and CMIP6 tables

**[orig #50]** List groundwater-related CSVs under `data/`  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/gw_framework/data")

print("Groundwater-related CSV files:")
for p in DATA.rglob("*.csv"):
    if any(k in str(p).lower() for k in [
        "groundwater", "water_level", "well", "gwl"
    ]):
        print(p)

**[orig #70]** Inspect groundwater parquet/CSV and main CMIP6 CSV products  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import xarray as xr

ROOT = Path("/content/drive/MyDrive/gw_framework/data")

# Inspect groundwater CSV/parquet files
for folder in ["gwl/processed", "gwl/processed/_parsed"]:
    path = ROOT / folder
    if not path.exists():
        continue

    print("\n###", folder)

    for f in sorted(path.iterdir()):
        if f.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(f, nrows=5)
                print("\nFILE:", f.name)
                print("Columns:", df.columns.tolist())
                display(df.head(3))
            except Exception as e:
                print("CSV error:", f.name, e)

        elif f.suffix.lower() == ".parquet":
            try:
                df = pd.read_parquet(f)
                print("\nFILE:", f.name)
                print("Shape:", df.shape)
                print("Columns:", df.columns.tolist())
                display(df.head(3))
            except Exception as e:
                print("Parquet error:", f.name, e)

# Inspect the main CMIP6 CSV products
cmip = ROOT / "Cmip6 indices" / "CMIP6 Indices"

candidates = [
    cmip / "Tmax" / "Raw Data" / "Tmax_Hybrid_SSP245_02112023.csv",
    cmip / "Tmin" / "Raw Data" / "Tmin_Hybrid_SSP245_02112023.csv",
    cmip / "PCP" / "Raw Data" / "PCP_Hybrid_SSP245_02112023.csv",
    cmip / "PCP" / "Raw Data" / "PCP_Hybrid_SSP585_02112023.csv",
]

for f in candidates:
    if f.exists():
        print("\nCMIP6 FILE:", f.name)
        df = pd.read_csv(f, nrows=5)
        print("Columns:", df.columns.tolist()[:20])
        print("Sample:")
        display(df.head(3))
    else:
        print("\nNot found:", f)

**[orig #71]** Row counts, well coordinates and CMIP6 date columns  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/gw_framework/data")

# 1. Inspect groundwater files and their actual row counts
print("=" * 60)
print("GROUNDWATER DATA DIAGNOSTICS")
print("=" * 60)

for folder in ["gwl/processed", "gwl/processed/_parsed"]:
    p = ROOT / folder
    if not p.exists():
        print("Missing folder:", p)
        continue

    for f in sorted(p.glob("*")):
        if f.suffix.lower() not in [".csv", ".parquet"]:
            continue
        try:
            if f.suffix.lower() == ".csv":
                d = pd.read_csv(f)
            else:
                d = pd.read_parquet(f)

            print(f"\n{f.relative_to(ROOT)}")
            print("Rows:", len(d), "| Columns:", list(d.columns))

            if len(d):
                display(d.head(3))

                for c in ["date", "dtw", "well_id"]:
                    if c in d.columns:
                        print(c, "non-null:", d[c].notna().sum())

                if "date" in d.columns:
                    dates = pd.to_datetime(d["date"], errors="coerce")
                    print("Date range:", dates.min(), "to", dates.max())

        except Exception as e:
            print("ERROR:", f.name, str(e))


# 2. Check coordinates of parsed wells
meta_path = ROOT / "gwl/processed/_parsed/895eea4cddcbde00_meta.parquet"

if meta_path.exists():
    meta = pd.read_parquet(meta_path)

    print("\n" + "=" * 60)
    print("WELL COORDINATE EXTENT")
    print("=" * 60)

    print("Number of wells:", len(meta))
    print("Columns:", meta.columns.tolist())

    if {"lat", "lon"}.issubset(meta.columns):
        meta["lat"] = pd.to_numeric(meta["lat"], errors="coerce")
        meta["lon"] = pd.to_numeric(meta["lon"], errors="coerce")

        print("Valid coordinates:",
              meta[["lat", "lon"]].dropna().shape[0])

        display(meta[[
            c for c in [
                "well_id", "state", "district",
                "lat", "lon", "n", "source"
            ] if c in meta.columns
        ]])

        print("Latitude range:", meta["lat"].min(), "to", meta["lat"].max())
        print("Longitude range:", meta["lon"].min(), "to", meta["lon"].max())


# 3. Verify the CMIP6 CSV date columns and sizes
print("\n" + "=" * 60)
print("CMIP6 DATE-COLUMN CHECK")
print("=" * 60)

cmip_dir = ROOT / "Cmip6 indices" / "CMIP6 Indices"

for variable in ["Tmax", "Tmin", "PCP"]:
    folder = cmip_dir / variable / "Raw Data"

    if not folder.exists():
        print("Folder not found:", folder)
        continue

    for f in sorted(folder.glob("*.csv")):
        try:
            head = pd.read_csv(f, nrows=3)
            print(f"\nFile: {f.name}")
            print("Size (MB):", round(f.stat().st_size / 1e6, 2))
            print("First 8 columns:", head.columns.tolist()[:8])
            print("First 3 rows:")
            display(head.iloc[:, :6].head(3))
        except Exception as e:
            print("ERROR:", f.name, str(e))

print("\nDiagnostic complete.")

**[orig #72]** Groundwater observation quality issues + CMIP6 checks  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/gw_framework/data")
PARSED = ROOT / "gwl/processed/_parsed"
OUT = ROOT / "gwl/diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# A. Inspect groundwater observations and quality issues
# -------------------------------------------------------
manual_files = list(PARSED.glob("*_man.parquet"))

if not manual_files:
    raise FileNotFoundError("No parsed manual groundwater parquet file found.")

manual = pd.concat(
    [pd.read_parquet(f) for f in manual_files],
    ignore_index=True
)

manual["date"] = pd.to_datetime(manual["date"], errors="coerce")
manual["dtw"] = pd.to_numeric(manual["dtw"], errors="coerce")

print("\nGROUNDWATER SUMMARY")
print("Rows:", len(manual))
print("Wells:", manual["well_id"].nunique())
print("Date range:", manual["date"].min(), "to", manual["date"].max())
print("Missing dates:", manual["date"].isna().sum())
print("Missing DTW:", manual["dtw"].isna().sum())
print("Duplicate well-date pairs:",
      manual.duplicated(["well_id", "date"]).sum())

print("\nDTW DISTRIBUTION")
print(manual["dtw"].describe(percentiles=[.01, .05, .5, .95, .99]))
print("\nDry-status counts:")
print(manual["dry"].value_counts(dropna=False))

print("\nObservations by well:")
print(manual.groupby("well_id").agg(
    observations=("dtw", "count"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    median_dtw=("dtw", "median")
).to_string())

# Save an audit copy, not a replacement for QC data.
manual.to_csv(OUT / "manual_observations_audit.csv", index=False)

# -------------------------------------------------------
# B. Inspect the original source file, if present
# -------------------------------------------------------
source = ROOT / "inputs/gwl/wells_levels_user_original.csv"

if source.exists():
    original = pd.read_csv(source)
    print("\nORIGINAL SOURCE FILE")
    print("Rows:", len(original))
    print("Columns:", original.columns.tolist())
    print(original.head(5).to_string(index=False))
else:
    print("\nOriginal CSV not found at:", source)

# -------------------------------------------------------
# C. Check well metadata
# -------------------------------------------------------
meta_files = list(PARSED.glob("*_meta.parquet"))

if meta_files:
    meta = pd.concat(
        [pd.read_parquet(f) for f in meta_files],
        ignore_index=True
    )

    print("\nWELL METADATA")
    print("Wells:", len(meta))
    print("Columns:", meta.columns.tolist())

    if {"lat", "lon"}.issubset(meta.columns):
        meta["lat"] = pd.to_numeric(meta["lat"], errors="coerce")
        meta["lon"] = pd.to_numeric(meta["lon"], errors="coerce")

        print("Latitude range:", meta["lat"].min(), meta["lat"].max())
        print("Longitude range:", meta["lon"].min(), meta["lon"].max())
        print(meta[["well_id", "lat", "lon"]].to_string(index=False))

# -------------------------------------------------------
# D. Audit CMIP6 files and date alignment
# -------------------------------------------------------
cmip = ROOT / "Cmip6 indices" / "CMIP6 Indices"

print("\nCMIP6 FILE AUDIT")

for variable in ["Tmax", "Tmin", "PCP"]:
    folder = cmip / variable / "Raw Data"

    if not folder.exists():
        print("MISSING FOLDER:", folder)
        continue

    for f in sorted(folder.glob("*.csv")):
        try:
            # Read the header and row count without loading
            # the entire file into memory.
            head = pd.read_csv(f, nrows=3)
            nrows = sum(1 for _ in open(f, encoding="utf-8")) - 1

            date_cols = [
                c for c in head.columns
                if str(c).strip().lower() in ["date", "datetime", "time"]
            ]

            print({
                "variable": variable,
                "file": f.name,
                "rows": nrows,
                "columns": len(head.columns),
                "date_column": date_cols,
                "first_columns": head.columns.tolist()[:5]
            })

            if date_cols:
                full_dates = pd.read_csv(
                    f, usecols=[date_cols[0]]
                )[date_cols[0]]

                parsed_dates = pd.to_datetime(
                    full_dates, errors="coerce", dayfirst=True
                )

                print(
                    "  Parsed dates:",
                    parsed_dates.notna().sum(),
                    "/",
                    len(parsed_dates),
                    "| Range:",
                    parsed_dates.min(),
                    "to",
                    parsed_dates.max()
                )

        except Exception as e:
            print("ERROR:", f.name, repr(e))

print("\nAudit complete. Files saved in:", OUT)


**[orig #145]** Search `data/` for climate and groundwater tables and print their columns  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/gw_framework/data")

print("Searching for climate and groundwater data files...")

extensions = {".csv", ".xlsx", ".xls", ".parquet", ".nc", ".nc4"}

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    name = path.name.lower()
    if any(term in name for term in [
        "cmip", "ssp", "rain", "precip", "pr_",
        "tasmax", "tasmin", "tmax", "tmin",
        "groundwater", "seasonal", "cleaned"
    ]):
        print(f"\nFILE: {path}")

        try:
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path, nrows=5)
                print("Columns:", df.columns.tolist())
                print(df.head(2).to_string(index=False))

            elif path.suffix.lower() in {".xlsx", ".xls"}:
                xls = pd.ExcelFile(path)
                print("Sheets:", xls.sheet_names)
                for sheet in xls.sheet_names[:2]:
                    sample = pd.read_excel(
                        path, sheet_name=sheet, nrows=3
                    )
                    print(f"Sheet {sheet} columns:",
                          sample.columns.tolist())

            elif path.suffix.lower() == ".parquet":
                df = pd.read_parquet(path)
                print("Columns:", df.columns.tolist())
                print(df.head(2).to_string(index=False))

            elif path.suffix.lower() in {".nc", ".nc4"}:
                try:
                    import xarray as xr
                    ds = xr.open_dataset(path)
                    print(ds)
                    ds.close()
                except Exception as exc:
                    print("NetCDF inspection issue:", exc)

        except Exception as exc:
            print("Could not inspect:", exc)

print("\nInventory complete. No source files were modified.")
